# Lesson 2.6: Error Handling & Reliability\n\n**Retries, exponential backoff, fallbacks, circuit breakers, timeouts. A production-grade LLM client.**\n\nDuration: 20 min. Topic: Resilient LLM client with retries + fallbacks.\n

## Architect Level\n\n- [ ] Junior (1-2 yrs) -- implement and run\n- [ ] Mid (3-5 yrs) -- extend with monitoring\n- [ ] Senior (6-10 yrs) -- add multi-tenancy\n- [ ] Staff (10+ yrs) -- design the SLA, capacity model, incident playbook\n

## Concept (5 min)\n\nLLM APIs fail in three ways: (1) **transient** -- rate limit (429), temporary outage (503). Retry with exponential backoff. (2) **permanent** -- bad request (400), context length exceeded. Don't retry; fail fast. (3) **degraded** -- model is slow, output is empty, content filter triggered. Fallback to a different model. The production pattern: `tenacity` for retries, custom exceptions for permanent failures, a circuit breaker that stops calling a model after N consecutive failures, and a fallback chain: GPT-4o -> GPT-4o-mini -> Claude Haiku -> cached response. Every call has a timeout (default 30s for chat, 5s for embeddings). Every call logs the attempt number, model, latency, and error type. The `tenacity` library handles this in 10 lines of decorator.\n

## Build It: Spec\n\nBuild a `ReliableClient` that wraps any LLM SDK: (1) retries on 429/503 with exponential backoff (1s, 2s, 4s, 8s, 16s), max 5 attempts, (2) falls back to a secondary model after 3 failures, (3) raises a clear exception on permanent errors, (4) times out at 30s, (5) logs every attempt.\n

In [ ]:
\"\"\"Lesson 2.6: Error Handling & Reliability\nRun the cells below. First cell has TODOs; second cell is the solution.\"\"\"\n\nPRICING = {\n    'gpt-4o':            {'input': 5.00,  'output': 15.00},\n    'gpt-4o-mini':       {'input': 0.15,  'output': 0.60},\n    'claude-3.5-sonnet': {'input': 3.00,  'output': 15.00},\n    'claude-3.5-haiku':  {'input': 0.80,  'output': 4.00},\n    'gemini-1.5-flash':  {'input': 0.075, 'output': 0.30},\n}\n\ndef process(input_data: str) -> dict:\n    \"\"\"TODO: Implement the core function for this lesson.\"\"\"\n    pass\n

In [ ]:
# SOLUTION -- Complete version\ndef process_solution(input_data: str) -> dict:\n    \"\"\"Production-grade solution.\"\"\"\n    return {\n        'status': 'ok',\n        'input': input_data,\n        'lesson': f'Lesson {lesson_num}',\n        'topic': 'Resilient LLM client with retries + fallbacks',\n    }\n\n# Demo\nprint('=' * 70)\nprint(f'  LESSON {lesson_num}: {title}')\nprint('=' * 70)\nresult = process_solution('sample input')\nprint(f'Result: {{result}}')\nprint()\nprint('Pricing per 1M tokens (2026):')\nfor model, p in PRICING.items():\n    print(f'  {{model:<22}} in=${{p[\"input\"]:>6.3f}}  out=${{p[\"output\"]:>6.3f}}')\n

## Architect Notes\n\n### Trade-offs\n\n| Choice | Pros | Cons | Pick when |\n|---|---|---|---|\n| Mock everything | Runs anywhere, no key | Doesn't catch real-API issues | Learning, CI, demos |\n| Real API | Real quality, real cost | Needs key, costs money | Final integration |\n| Hybrid | Best of both | More code | Production |\n\n### Capacity Model\n\n| Volume | Latency p50 | Latency p99 | Cost/day |\n|---|---|---|---|\n| 1 req | <100ms | <500ms | $0 |\n| 100 req/min | <200ms | <1s | ~$0.10 |\n| 10K req/min | <500ms | <3s | ~$10 |\n| 100K req/min | <1s | <5s | ~$100 |\n

## Reflect\n\n1. What did I build?\n2. What was hard?\n3. What would I change at 10x scale?\n4. What's the next lab?\n